# NPS Agent — OpenAI Direct (Agents SDK)

Calls OpenAI's gpt-4o directly using the [OpenAI Agents SDK](https://github.com/openai/openai-agents-python) with MCP tools, for latency/token benchmarking against the Llama Stack variant (`nps_agent.ipynb`).

**Prerequisites:**
- NPS MCP server running on `localhost:3005`
- `OPENAI_API_KEY` in environment
- `openai-agents` package installed

In [43]:
import os
import time
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

import mlflow
from mlflow.entities import SpanType, AssessmentSource, AssessmentSourceType
from mlflow.genai.judges import make_judge
from agents import Agent, Runner
from agents.mcp import MCPServerSse
from typing import Literal

# Auto-instrument OpenAI Agents SDK calls for MLflow tracing
mlflow.openai.autolog()

## Configuration

In [44]:
# MCP URL uses localhost (no container boundary — the Agents SDK runs the MCP client locally)
# No trailing slash — the SSE client doesn't follow redirects
NPS_MCP_URL = "http://localhost:3005/sse"
MODEL_ID = "gpt-4o"
JUDGE_MODEL = "openai:/gpt-4o"

AGENT_INSTRUCTIONS = (
    "You are a helpful National Parks Service assistant. "
    "Use the available tools to answer questions about national parks, "
    "events, activities, campgrounds, and visitor information."
)

In [45]:
db_path = os.path.join(os.getcwd(), "mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("nps-agent")
print(f"MLflow database: {db_path}")

MLflow database: /Users/sschifma/DEV/rh_repos/agents/examples/agents_tracing-eval_mlflow/nps_agent/mlflow.db


## Agent Function

Uses the OpenAI Agents SDK with `MCPServerSse` to connect to the already-running NPS MCP server.
The SDK handles the tool-calling loop internally via `Runner.run()`.
`mlflow.openai.autolog()` captures spans for each model call, tool call, and handoff automatically.

In [46]:
from agents import RunHooks
from agents.run_context import RunContextWrapper

class UsageTracker(RunHooks):
    """Accumulates token usage across all LLM calls in a run."""
    def __init__(self):
        self.input_tokens = 0
        self.output_tokens = 0
        self.total_tokens = 0

    async def on_llm_end(self, context, agent, response):
        if response.usage:
            self.input_tokens += response.usage.input_tokens
            self.output_tokens += response.usage.output_tokens
            self.total_tokens += response.usage.total_tokens


async def run_nps_agent(prompt: str, model: str = MODEL_ID) -> str:
    """Run the NPS agent via OpenAI Agents SDK with MCP tools."""
    async with MCPServerSse(
        params={"url": NPS_MCP_URL},
        name="NPS MCP Server",
    ) as mcp_server:
        agent = Agent(
            name="NPS Agent",
            instructions=AGENT_INSTRUCTIONS,
            mcp_servers=[mcp_server],
            model=model,
        )

        tracker = UsageTracker()
        start_time = time.time()
        result = await Runner.run(agent, prompt, hooks=tracker)
        latency_ms = (time.time() - start_time) * 1000

        mlflow.log_metrics({
            "latency_ms": latency_ms,
            "input_tokens": tracker.input_tokens,
            "output_tokens": tracker.output_tokens,
            "total_tokens": tracker.total_tokens,
        })

        return result.final_output

## Agent-as-a-Judge

In [47]:
nps_judge = make_judge(
    name="nps_agent_evaluator",
    instructions=(
        "Evaluate the NPS agent's performance in {{ trace }}.\n\n"
        "Check for:\n"
        "1. Response Quality: Did the agent correctly identify parks and provide accurate information?\n"
        "2. Tool Usage: Were the correct NPS MCP tools used (search_parks, get_park_events, etc.)?\n"
        "3. Completeness: Did the agent answer all parts of the user's question?\n\n"
        "Rate as: 'good', 'acceptable', or 'poor'"
    ),
    feedback_value_type=Literal["good", "acceptable", "poor"],
    model=JUDGE_MODEL,
)

In [48]:
def evaluate_trace(trace):
    """Run Agent-as-a-Judge evaluation and log to MLflow."""
    feedback = nps_judge(trace=trace)

    trace_id = trace.info.trace_id
    mlflow.log_feedback(
        trace_id=trace_id,
        name="nps_agent_evaluation",
        value=feedback.value,
        rationale=feedback.rationale,
        source=AssessmentSource(
            source_type=AssessmentSourceType.LLM_JUDGE,
            source_id=f"agent-as-a-judge/{JUDGE_MODEL}",
        ),
    )

    print(f"\nEvaluation: {feedback.value}")
    print(f"Rationale: {feedback.rationale}")
    return feedback

## Run Agent & Evaluate

In [49]:
prompt = "Tell me about some parks in Rhode Island, and let me know if there are any upcoming events at them."

mlflow.start_run(run_name=f"nps-openai-agent-sdk-{MODEL_ID}")
try:
    result = await run_nps_agent(prompt)
    print(f"Response:\n{result}")

    mlflow.flush_trace_async_logging()

    trace_id = mlflow.get_last_active_trace_id()
    trace = mlflow.get_trace(trace_id)
    evaluate_trace(trace)
finally:
    mlflow.end_run()

Response:
Here are some national parks in Rhode Island and their upcoming events:

### Blackstone River Valley National Historical Park
- **Location:** Pawtucket, RI
- **Website:** [Blackstone River Valley NHP](https://www.nps.gov/blrv/index.htm)
- **Description:** This park explores the origins of the American Industrial Revolution and its impact, starting with Samuel Slater's cotton spinning mill.

#### Upcoming Events:
1. **Old Slater Mill Tour:** Join a Park Ranger on a tour of the Mill.
2. **First Strike Commemoration:** Events include hands-on activities and weaving workshops.
3. **Take Me Fishing:** Learn to fish with park staff.
4. **Ranger Walkabouts and Talks:** Various ranger-led walks and talks throughout the summer.
5. **Revolutionary War Veteran's Pension Transcription Program**

### Roger Williams National Memorial
- **Location:** Providence, RI
- **Website:** [Roger Williams NM](https://www.nps.gov/rowi/index.htm)
- **Description:** This site commemorates the founder of

## View Traces in MLflow UI

```bash
mlflow ui --port 5001
```

Open http://localhost:5001 in your browser.

### Comparing Runs

1. Select the `nps-agent` experiment
2. Check both runs: `nps-openai/gpt-4o` (Llama Stack) and `nps-openai-direct-gpt-4o` (OpenAI direct)
3. Click **Compare** to see latency and token metrics side-by-side

The `mlflow.openai.autolog()` integration automatically captures token usage in the trace spans.
The `latency_ms` metric is logged explicitly for run-level comparison.